# RAG Assignment: Baseline Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking

**Document:** Regulation (EU) 2016/679 – *General Data Protection Regulation (GDPR)*, official EUR-Lex PDF (88 pages, 173 recitals + 99 articles).
It is a good stress test because it is full of general rules followed by *"shall not apply where…"* exceptions, prohibitions,
role-specific restrictions, and recitals that paraphrase the articles (natural lexical distractors).

| Component | Choice |
|---|---|
| Bi-Encoder (stage 1) | `sentence-transformers/all-MiniLM-L6-v2` (384-d, normalised → inner product = cosine) |
| Vector store | in-memory exact inner-product index (NumPy matrix, equivalent to FAISS `IndexFlatIP`) |
| Cross-Encoder (stage 2) | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| LLM | `Qwen/Qwen2.5-3B-Instruct` (local open-weights, fp16 on GPU, greedy decoding → deterministic) |

**Pipeline 1 (Baseline):** query → bi-encoder → Top-10 by cosine → Top-3 → LLM → *Answer 1*
**Pipeline 2 (Re-rank):** same Top-10 → Cross-Encoder scores all 10 (query, chunk) pairs → re-sort → Top-3 → *same prompt* → *Answer 2*

Both pipelines share the exact same chunk corpus and the exact same first-stage candidate list.

## 0. Setup & configuration

In [1]:
# If running on a fresh machine:
# pip install torch sentence-transformers transformers accelerate pymupdf numpy pandas
import os, re, time, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import pymupdf  # PyMuPDF, for PDF text extraction with font information
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---- configuration ----
PDF_PATH   = Path("data/gdpr.pdf")
PDF_URL    = "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679"
BI_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENC  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# local path if the weights were downloaded manually, otherwise the HF hub id
LLM_NAME   = "models/Qwen2.5-3B-Instruct" if Path("models/Qwen2.5-3B-Instruct/config.json").exists() else "Qwen/Qwen2.5-3B-Instruct"
TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_K_CONTEXT  = 3    # chunks given to the LLM
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

device: cuda NVIDIA GeForce RTX 4070 Laptop GPU


## 1. Load the PDF and clean the text

The Official Journal layout has a running header on every page (`4.5.2016 | EN | Official Journal… | L 119/x`) and
footnotes at the bottom of pages. Body text is set in 9.6 pt, footnotes in 8.5 pt and footnote markers in superscript (< 6 pt),
so I use PyMuPDF's span-level font sizes to keep only body text, and a regex to drop the running header.

In [2]:
if not PDF_PATH.exists():  # download the official PDF if it is not in the repo
    import urllib.request
    PDF_PATH.parent.mkdir(exist_ok=True)
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    PDF_PATH.write_bytes(urllib.request.urlopen(req).read())

HEADER_RE = re.compile(r"\d{1,2}\.\d{1,2}\.2016|EN|Official Journal of the European Union|L \d+/\d+")

def load_pages(path):
    # Returns [(page_number, cleaned_text)], one entry per page.
    doc = pymupdf.open(path)
    pages = []
    for i, page in enumerate(doc):
        lines = []
        for blk in page.get_text("dict")["blocks"]:
            for ln in blk.get("lines", []):
                # keep only body-size spans (drops footnotes + superscript footnote markers)
                s = "".join(sp["text"] for sp in ln["spans"] if sp["size"] >= 9).strip()
                if s and not HEADER_RE.fullmatch(s):
                    lines.append(s)
        pages.append((i + 1, "\n".join(lines)))
    return pages

pages = load_pages(PDF_PATH)
print(f"{len(pages)} pages, {sum(len(t.split()) for _, t in pages):,} words")
print(pages[35][1][:700])

88 pages, 54,104 words
(e)  kept in a form which permits identification of data subjects for no longer than is necessary for the purposes for
which the personal data are processed; personal data may be stored for longer periods insofar as the personal data
will be processed solely for archiving purposes in the public interest, scientific or historical research purposes or
statistical purposes in accordance with Article 89(1) subject to implementation of the appropriate technical and
organisational measures required by this Regulation in order to safeguard the rights and freedoms of the data subject
(‘storage limitation’);
(f)  processed in a manner that ensures appropriate security of the personal data, including 


## 2. Chunking

Strategy: **structure-aware packing**.
1. Split the text into paragraphs at the document's natural boundaries: numbered recitals `(n)`, numbered article
   paragraphs `n.`, `Article n` headings, chapter and section headings.
2. Track which Recital / Article each paragraph belongs to (metadata only, used for display and analysis; it is **not**
   prepended to the text, so both retrievers see exactly the same raw text).
3. Greedily pack consecutive paragraphs into chunks of ≤ 180 words (≈ 250 tokens, well inside the cross-encoder's 512-token
   limit). A short trailing paragraph is repeated at the start of the next chunk as overlap; a paragraph longer than 180
   words is split with a 40-word sliding overlap.

In [3]:
SPLIT_RE = re.compile(r"\n(?=\(\d+\)\s|\d+\.\s|Article \d+\n|CHAPTER [IVX]+\n|Section \d+\n)")

def split_paragraphs(pages):
    paras, section, last_art, last_rec = [], "Preamble", 0, 0
    for pno, txt in pages:
        for block in SPLIT_RE.split(txt):
            b = re.sub(r"\s*\n\s*", " ", block).strip()
            b = re.sub(r"­\s*", "", b)          # remove soft hyphens from justified text
            if not b:
                continue
            m = re.match(r"Article (\d+)\b", b)
            r = re.match(r"\((\d+)\)\s", b)
            # section numbers only move forward, so cross-references like "(1)" inside text do not confuse the tracker
            if m and int(m.group(1)) >= last_art:
                last_art = int(m.group(1)); section = f"Article {last_art}"
            elif r and last_art == 0 and int(r.group(1)) == last_rec + 1:
                last_rec = int(r.group(1)); section = f"Recital {last_rec}"
            paras.append({"page": pno, "section": section, "text": b})
    return paras

def build_chunks(paras, max_words=180, overlap_words=40):
    chunks, cur, cur_words = [], [], 0

    def flush(group):
        if group:
            sec = group[0]["section"] if group[0]["section"] == group[-1]["section"] \
                  else f'{group[0]["section"]} – {group[-1]["section"]}'
            chunks.append({"id": len(chunks), "page": group[0]["page"], "section": sec,
                           "text": " ".join(p["text"] for p in group)})

    for p in paras:
        w = len(p["text"].split())
        if w > max_words:                        # very long paragraph -> sliding window
            flush(cur); cur, cur_words = [], 0
            words, step = p["text"].split(), max_words - overlap_words
            for s in range(0, len(words), step):
                flush([{**p, "text": " ".join(words[s:s + max_words])}])
                if s + max_words >= len(words):
                    break
            continue
        if cur_words + w > max_words:            # chunk full -> emit it, carry a short last paragraph as overlap
            flush(cur)
            last = cur[-1] if cur and len(cur[-1]["text"].split()) <= overlap_words else None
            cur = [last] if last else []
            cur_words = len(last["text"].split()) if last else 0
        cur.append(p); cur_words += w
    flush(cur)
    return chunks

chunks = build_chunks(split_paragraphs(pages))
chunk_texts = [c["text"] for c in chunks]
lens = np.array([len(t.split()) for t in chunk_texts])
print(f"{len(chunks)} chunks | words per chunk: mean {lens.mean():.0f}, min {lens.min()}, max {lens.max()}")
pd.DataFrame(chunks).iloc[[1, 100, 288, 410]]

430 chunks | words per chunk: mean 136, min 9, max 194


,id,page,section,text
1,1,1,Recital 1 – Recital 2,(1) The protection of natural persons in relation to the processing of personal data is a fundamental right. Article...
100,100,16,Recital 85,"(85) A personal data breach may, if not addressed in an appropriate and timely manner, result in physical, material ..."
288,288,53,Article 34,3. The communication to the data subject referred to in paragraph 1 shall not be required if any of the following co...
410,410,82,Article 83,"3. If a controller or processor intentionally or negligently, for the same or linked processing operations, infringe..."


## 3. Models

### 3.1 Bi-Encoder + vector store (Pipeline 1, stage 1)

The bi-encoder embeds every chunk **once, offline**, independently of any query. Embeddings are L2-normalised, so the inner
product equals cosine similarity. The index is an exact (brute-force) inner-product search over a NumPy matrix – for 430
vectors this is exact and takes well under a millisecond.

In [4]:
class VectorStore:
    # Minimal exact inner-product vector store (same behaviour as faiss.IndexFlatIP on normalised vectors).
    def __init__(self, embeddings):
        self.emb = embeddings.astype(np.float32)
    def search(self, q_emb, k):
        scores = self.emb @ q_emb                     # cosine similarity (vectors are unit-length)
        idx = np.argpartition(-scores, k)[:k]          # top-k without a full sort
        idx = idx[np.argsort(-scores[idx])]
        return idx, scores[idx], scores

bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)
t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
index_time = time.perf_counter() - t0
store = VectorStore(chunk_emb)
print(f"indexed {chunk_emb.shape[0]} chunks, dim={chunk_emb.shape[1]}, offline indexing time {index_time:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

indexed 430 chunks, dim=384, offline indexing time 0.61s


### 3.2 Cross-Encoder (Pipeline 2, stage 2)

In [5]:
cross_encoder = CrossEncoder(CROSS_ENC, device=DEVICE, max_length=512)
print(cross_encoder.predict([("What is the deadline for breach notification?", chunk_texts[0])]))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[-11.120734]


### 3.3 LLM + shared prompt template

The **same** prompt template and decoding settings are used for both pipelines; only the 3 context chunks differ.
Greedy decoding (`do_sample=False`) makes the answers deterministic, so any difference between Answer 1 and Answer 2 comes from
the retrieved context alone.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16, device_map=DEVICE)
llm.eval()

SYSTEM_PROMPT = (
    "You are a legal assistant answering questions about the EU General Data Protection Regulation (GDPR). "
    "Answer ONLY using the numbered context passages provided. Quote the exact figures, conditions and exceptions "
    "that the passages state, and cite the passage numbers you used like [1]. If the passages do not contain the "
    "answer, say \"The provided context does not contain the answer.\" Keep the answer under 120 words."
)

def build_prompt(query, ctx_chunks):
    context = "\n\n".join(f"[{i+1}] ({c['section']}) {c['text']}" for i, c in enumerate(ctx_chunks))
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context passages:\n{context}\n\nQuestion: {query}\nAnswer:"},
    ]

@torch.inference_mode()
def generate_answer(query, ctx_chunks, max_new_tokens=220):
    text = tokenizer.apply_chat_template(build_prompt(query, ctx_chunks), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                       temperature=None, top_p=None, top_k=None, pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    generate_answer.last_n_tokens = len(new_tokens)   # kept for latency analysis (time depends on answer length)
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(generate_answer("What does GDPR stand for?", [chunks[0]]))

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

GDPR stands for General Data Protection Regulation [1].


## 4. The two pipelines

In [7]:
def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    # Pipeline 1 retrieval: embed query with the bi-encoder, exact cosine search, return top-k.
    sync(); t0 = time.perf_counter()
    q = bi_encoder.encode(query, normalize_embeddings=True, convert_to_numpy=True)
    sync(); t1 = time.perf_counter()
    idx, scores, all_scores = store.search(q, k)
    t2 = time.perf_counter()
    return {"idx": idx, "bi_scores": scores, "all_scores": all_scores,
            "t_embed": t1 - t0, "t_search": t2 - t1, "t_total": t2 - t0}

def rerank(query, cand_idx):
    # Pipeline 2 stage 2: score every (query, candidate) pair jointly with the cross-encoder, sort descending.
    sync(); t0 = time.perf_counter()
    ce_scores = cross_encoder.predict([(query, chunk_texts[i]) for i in cand_idx], batch_size=len(cand_idx))
    order = np.argsort(-ce_scores)
    sync(); t1 = time.perf_counter()
    return {"idx": np.asarray(cand_idx)[order], "ce_scores": ce_scores[order], "t_rerank": t1 - t0,
            "ce_by_cand": dict(zip(map(int, cand_idx), map(float, ce_scores)))}

def run_both(query, with_llm=True):
    base = baseline_retrieve(query)
    rr = rerank(query, base["idx"])
    res = {"query": query, "base": base, "rr": rr}
    if with_llm:
        for name, idx in [("answer1", base["idx"][:TOP_K_CONTEXT]), ("answer2", rr["idx"][:TOP_K_CONTEXT])]:
            sync(); t0 = time.perf_counter()
            res[name] = generate_answer(query, [chunks[i] for i in idx])
            sync(); res["t_" + name] = time.perf_counter() - t0
            res["n_" + name] = generate_answer.last_n_tokens
    return res

### Helper: rank-shift table
For each query I mark the **critical chunk(s)** – the chunk(s) whose text actually contains the answer – using a regex on
the exact legal wording (e.g. `not later than 72 hours`). The table lists the 10 candidates with their bi-encoder rank/score,
cross-encoder score/rank, and whether they are critical. The ★ column marks critical chunks.

In [8]:
def gold_ids(pattern):
    return [c["id"] for c in chunks if re.search(pattern, c["text"])]

def rank_table(res, gold):
    base, rr = res["base"], res["rr"]
    ce_rank = {int(i): r + 1 for r, i in enumerate(rr["idx"])}
    rows = []
    for r, i in enumerate(base["idx"]):
        i = int(i)
        rows.append({"chunk": i, "section": chunks[i]["section"], "page": chunks[i]["page"],
                     "bi_rank": r + 1, "bi_score": round(float(base["bi_scores"][r]), 3),
                     "ce_score": round(rr["ce_by_cand"][i], 2), "ce_rank": ce_rank[i],
                     "★": "★" if i in gold else "", "text": chunk_texts[i][:110] + "…"})
    return pd.DataFrame(rows).sort_values("ce_rank").reset_index(drop=True)

def gold_shift(res, gold):
    # Bi-encoder rank over the full corpus and cross-encoder rank (only defined if the chunk is in the top-10)
    full_order = list(np.argsort(-res["base"]["all_scores"]))
    rr_list = [int(i) for i in res["rr"]["idx"]]
    return [{"chunk": g, "section": chunks[g]["section"],
             "baseline_rank": full_order.index(g) + 1,
             "reranked_rank": (rr_list.index(g) + 1) if g in rr_list else None} for g in gold]

def show(res, gold, category):
    print("=" * 110); print(f"[{category}]  Q: {res['query']}"); print("=" * 110)
    display(rank_table(res, gold))
    for s in gold_shift(res, gold):
        rr = f"#{s['reranked_rank']}" if s["reranked_rank"] else "not in top-10"
        print(f"Critical chunk {s['chunk']} ({s['section']}): Rank #{s['baseline_rank']} → {rr}")
    print("\nTop-3 given to LLM  | baseline :", [f"{int(i)} ({chunks[i]['section']})" for i in res["base"]["idx"][:3]])
    print("                    | re-ranked:", [f"{int(i)} ({chunks[i]['section']})" for i in res["rr"]["idx"][:3]])
    print("\n--- Answer 1 (baseline) ---\n" + res["answer1"])
    print("\n--- Answer 2 (re-ranked) ---\n" + res["answer2"])

## 5. Failure-mode analysis (stress tests)

Each query targets one failure category from the assignment. The regex identifies the critical chunk(s) that contain the
correct answer, so the rank shift is measured objectively rather than by eye.

| # | Category | Query idea | Why the bi-encoder is expected to struggle |
|---|---|---|---|
| 1 | Keyword trap / distractor | fine for *failing to report a data breach* | "fine" + "breach" overlap strongly with the €20M chunk and breach recitals; the correct €10M tier never says "breach" – it says "obligations … pursuant to Articles 25 to 39" |
| 2 | Exception / corner case | when is telling individuals about a breach *not* required | general notification-duty chunks dominate; the "shall not be required if…" paragraph is separate |
| 3 | Negation / prohibition | is it *forbidden* to process biometric data | the prohibition sits in Art 9(1) ("shall be prohibited"), but recitals that say biometric data "should not be processed, *unless*…" and chunks about *permitted* processing are lexically closer |
| 4 | Role limit | can a supervisory-authority member have *another paid job* | the binding rule is phrased "shall not … engage in any incompatible occupation, whether gainful or not" (negation + synonyms for "paid job") |
| 5 | Buried chunk (rank 4–10) | maximum fine for violating the basic principles | many fine-related chunks (Art 83 general conditions, recitals) outrank the specific €20M/4% paragraph |
| 6 | Counter-example | when does the right to be forgotten *not* apply | included to show a case where re-ranking does **not** help |
| 7 | Counter-example (recall ceiling) | can public authorities rely on legitimate interests | the critical sentence is outside the top-10, so no re-ranker can recover it |

In [9]:
FAILURE_QUERIES = [
    ("Keyword trap / distractor",
     "How much can a company be fined for failing to report a data breach?",
     r"10 000 000 EUR"),
    ("Exception / corner case",
     "When is a controller not required to tell affected individuals about a personal data breach?",
     r"communication to the data subject referred to in paragraph 1 shall not be required"),
    ("Negation / prohibition",
     "Is it forbidden to process biometric data to identify a person?",
     r"biometric data for the purpose of uniquely identifying"),
    ("Role limit (synonyms + negation)",
     "Can a member of a supervisory authority have another paid job?",
     r"incompatible occupation, whether gainful or not"),
    ("Buried chunk (rank 4-10)",
     "What is the maximum fine for violating the basic principles of processing?",
     r"basic principles for processing, including conditions for consent"),
    ("Counter-example: re-ranking hurts",
     "When does the right to be forgotten not apply?",
     r"Paragraphs 1 and 2 shall not apply to the extent that processing is necessary"),
    ("Counter-example: first-stage recall ceiling",
     "Can public authorities rely on legitimate interests as a legal basis?",
     r"shall not apply to processing carried out by public authorities in the performance of their tasks"),
]

# warm-up so that first-call CUDA initialisation does not distort anything
_ = run_both("warm-up query about consent", with_llm=False)

failure_results = []
for cat, q, pat in FAILURE_QUERIES:
    gold = gold_ids(pat)
    res = run_both(q)
    res.update(category=cat, gold=gold)
    failure_results.append(res)

In [10]:
show(failure_results[0], failure_results[0]['gold'], failure_results[0]['category'])

[Keyword trap / distractor]  Q: How much can a company be fined for failing to report a data breach?


,chunk,section,page,bi_rank,bi_score,ce_score,ce_rank,★,text
0,411,Article 83,83,1,0.540,0.92,1,,"5. Infringements of the following provisions shall, in accordance with paragraph 2, be subject to administrati…"
1,100,Recital 85,16,2,0.511,-3.70,2,,"(85) A personal data breach may, if not addressed in an appropriate and timely manner, result in physical, mat…"
2,410,Article 83,82,7,0.466,-3.72,3,★,"3. If a controller or processor intentionally or negligently, for the same or linked processing operations, in…"
3,104,Recital 88,17,4,0.496,-4.40,4,,(88) In setting detailed rules concerning the format and procedures applicable to the notification of personal…
4,412,Article 83,83,9,0.446,-4.46,5,,"6. Non-compliance with an order by the supervisory authority as referred to in Article 58(2) shall, in accorda…"
5,103,Recital 87,17,3,0.503,-4.97,6,,(87) It should be ascertained whether all appropriate technological protection and organisational measures hav…
6,287,Article 33 – Article 34,52,5,0.480,-5.19,7,,"4. Where, and in so far as, it is not possible to provide the information at the same time, the information ma…"
7,389,Article 70,77,6,0.472,-5.36,8,,"criteria and conditions for decisions based on profiling pursuant to Article 22(2); (g) issue guidelines, reco…"
8,331,Article 47,63,10,0.442,-8.68,9,,it proves that that member is not responsible for the event giving rise to the damage; (g) how the information…
9,351,Article 57,68,8,0.464,-9.53,10,,"lodged by a data subject, or by a body, organisation or association in accordance with Article 80, and investi…"


Critical chunk 410 (Article 83): Rank #7 → #3

Top-3 given to LLM  | baseline : ['411 (Article 83)', '100 (Recital 85)', '103 (Recital 87)']
                    | re-ranked: ['411 (Article 83)', '100 (Recital 85)', '410 (Article 83)']

--- Answer 1 (baseline) ---
A company can be fined up to 4% of its global annual turnover or 20 million euros, whichever is greater, for failing to report a data breach [1].

--- Answer 2 (re-ranked) ---
The provided context does not contain the answer. [1]


In [11]:
show(failure_results[1], failure_results[1]['gold'], failure_results[1]['category'])

[Exception / corner case]  Q: When is a controller not required to tell affected individuals about a personal data breach?


,chunk,section,page,bi_rank,bi_score,ce_score,ce_rank,★,text
0,101,Recital 85,17,3,0.726,6.84,1,,"aware that a personal data breach has occurred, the controller should notify the personal data breach to the s…"
1,288,Article 34,53,5,0.714,6.79,2,★,3. The communication to the data subject referred to in paragraph 1 shall not be required if any of the follow…
2,285,Article 33,52,7,0.665,5.48,3,,Article 33 Notification of a personal data breach to the supervisory authority 1. In the case of a personal da…
3,289,Article 34 – Article 35,53,10,0.622,5.44,4,,"4. If the controller has not already communicated the personal data breach to the data subject, the supervisor…"
4,100,Recital 85,16,1,0.760,5.16,5,,"(85) A personal data breach may, if not addressed in an appropriate and timely manner, result in physical, mat…"
5,286,Article 33,52,4,0.721,5.00,6,,2. The processor shall notify the controller without undue delay after becoming aware of a personal data breac…
6,287,Article 33 – Article 34,52,6,0.699,4.61,7,,"4. Where, and in so far as, it is not possible to provide the information at the same time, the information ma…"
7,102,Recital 86,17,2,0.751,4.59,8,,"(86) The controller should communicate to the data subject a personal data breach, without undue delay, where …"
8,103,Recital 87,17,9,0.627,0.76,9,,(87) It should be ascertained whether all appropriate technological protection and organisational measures hav…
9,104,Recital 88,17,8,0.647,0.41,10,,(88) In setting detailed rules concerning the format and procedures applicable to the notification of personal…


Critical chunk 288 (Article 34): Rank #5 → #2

Top-3 given to LLM  | baseline : ['100 (Recital 85)', '102 (Recital 86)', '101 (Recital 85)']
                    | re-ranked: ['101 (Recital 85)', '288 (Article 34)', '285 (Article 33)']

--- Answer 1 (baseline) ---
A controller is not required to notify affected individuals about a personal data breach if they can demonstrate, in accordance with the accountability principle, that the personal data breach is unlikely to result in a risk to the rights and freedoms of natural persons [3].

--- Answer 2 (re-ranked) ---
A controller is not required to tell affected individuals about a personal data breach if any of the following conditions are met: (a) the controller has implemented appropriate technical and organizational protection measures, and those measures were applied to the personal data affected by the personal data breach, in particular those that render the personal data unintelligible to any person who is not authorized to access 

In [12]:
show(failure_results[2], failure_results[2]['gold'], failure_results[2]['category'])

[Negation / prohibition]  Q: Is it forbidden to process biometric data to identify a person?


,chunk,section,page,bi_rank,bi_score,ce_score,ce_rank,★,text
0,54,Recital 51,10,1,0.595,2.87,1,,"(51) Personal data which are, by their nature, particularly sensitive in relation to fundamental rights and fr…"
1,223,Article 8 – Article 9,38,8,0.488,2.71,2,★,2. The controller shall make reasonable efforts to verify in such cases that consent is given or authorised by…
2,62,Recital 57,11,2,0.585,1.82,3,,"(57) If the personal data processed by a controller do not permit the controller to identify a natural person,…"
3,229,Article 9 – Article 11,39,3,0.560,1.63,4,,"4. Member States may maintain or introduce further conditions, including limitations, with regard to the proce…"
4,206,Article 4,34,5,0.497,0.13,5,,"(12) ‘personal data breach’ means a breach of security leading to the accidental or unlawful destruction, loss…"
5,23,Recital 26,5,4,0.557,-1.39,6,,(26) The principles of data protection should apply to any information concerning an identified or identifiabl…
6,79,Recital 71,14,6,0.495,-3.26,7,,"(71) The data subject should have the right not to be subject to a decision, which may include a measure, eval…"
7,25,Recital 27 – Recital 29,5,9,0.485,-5.59,8,,(27) This Regulation does not apply to the personal data of deceased persons. Member States may provide for ru…
8,220,Article 6,37,7,0.491,-6.82,9,,"of personal data are processed, pursuant to Article 9, or whether personal data related to criminal conviction…"
9,26,Recital 30,6,10,0.484,-7.83,10,,"(30) Natural persons may be associated with online identifiers provided by their devices, applications, tools …"


Critical chunk 223 (Article 8 – Article 9): Rank #8 → #2

Top-3 given to LLM  | baseline : ['54 (Recital 51)', '62 (Recital 57)', '229 (Article 9 – Article 11)']
                    | re-ranked: ['54 (Recital 51)', '223 (Article 8 – Article 9)', '62 (Recital 57)']

--- Answer 1 (baseline) ---
Based on [1], biometric data should not be processed through a specific technical means allowing the unique identification or authentication of a natural person, unless processing is allowed in specific cases set out in this Regulation. Therefore, it is not forbidden but requires specific authorization. [1]

--- Answer 2 (re-ranked) ---
Yes, according to [2], biometric data should not be processed for the purpose of uniquely identifying a natural person unless allowed by specific provisions in this Regulation. Exceptions may exist under Article 9, paragraph 1, which allows processing if it is for purposes related to health or sex life/sexual orientation, but these are not for the purpose of unique

In [13]:
show(failure_results[3], failure_results[3]['gold'], failure_results[3]['category'])

[Role limit (synonyms + negation)]  Q: Can a member of a supervisory authority have another paid job?


,chunk,section,page,bi_rank,bi_score,ce_score,ce_rank,★,text
0,343,Article 52 – Article 53,66,7,0.577,1.84,1,★,3. Member or members of each supervisory authority shall refrain from any action incompatible with their dutie…
1,142,Recital 121,22,1,0.645,1.71,2,★,(121) The general conditions for the member or members of the supervisory authority should be laid down by la…
2,344,Article 53 – Article 54,66,3,0.622,1.00,3,,Article 53 General conditions for the members of the supervisory authority 1. Member States shall provide for …
3,372,Article 62,73,2,0.628,0.55,4,,"3. A supervisory authority may, in accordance with Member State law, and with the seconding supervisory author…"
4,140,Recital 117 – Recital 119,22,6,0.586,0.37,5,,"(117) The establishment of supervisory authorities in Member States, empowered to perform their tasks and exe…"
5,345,Article 54,66,4,0.615,-0.35,6,,1. Each Member State shall provide by law for all of the following: (a) the establishment of each supervisory…
6,342,Article 51 – Article 52,65,10,0.561,-0.82,7,★,4. Each Member State shall notify to the Commission the provisions of its law which it adopts pursuant to this…
7,153,Recital 129,24,8,0.568,-1.26,8,,"supervisory authority authorised by him or her, give the reasons for the measure, and refer to the right of an…"
8,209,Article 4,34,9,0.564,-1.91,9,,(21) ‘supervisory authority’ means an independent public authority which is established by a Member State purs…
9,361,Article 58,70,5,0.606,-2.29,10,,4. The exercise of the powers conferred on the supervisory authority pursuant to this Article shall be subject…


Critical chunk 142 (Recital 121): Rank #1 → #2
Critical chunk 342 (Article 51 – Article 52): Rank #10 → #7
Critical chunk 343 (Article 52 – Article 53): Rank #7 → #1

Top-3 given to LLM  | baseline : ['142 (Recital 121)', '372 (Article 62)', '344 (Article 53 – Article 54)']
                    | re-ranked: ['343 (Article 52 – Article 53)', '142 (Recital 121)', '344 (Article 53 – Article 54)']

--- Answer 1 (baseline) ---
No, a member of a supervisory authority cannot engage in any incompatible occupation during their term of office, whether gainful or not [1].

--- Answer 2 (re-ranked) ---
No, according to [2] 4., a member of the supervisory authority shall not, during their term of office, engage in any incompatible occupation, whether gainful or not.


In [14]:
show(failure_results[4], failure_results[4]['gold'], failure_results[4]['category'])

[Buried chunk (rank 4-10)]  Q: What is the maximum fine for violating the basic principles of processing?


,chunk,section,page,bi_rank,bi_score,ce_score,ce_rank,★,text
0,411,Article 83,83,8,0.558,5.52,1,★,"5. Infringements of the following provisions shall, in accordance with paragraph 2, be subject to administrati…"
1,410,Article 83,82,1,0.685,1.22,2,,"3. If a controller or processor intentionally or negligently, for the same or linked processing operations, in…"
2,408,Article 83,82,3,0.586,-2.04,3,,"2. Administrative fines shall, depending on the circumstances of each individual case, be imposed in addition …"
3,175,Recital 148,27,2,0.614,-2.59,4,,"(148) In order to strengthen the enforcement of the rules of this Regulation, penalties including administrat…"
4,414,Article 84 – Article 85,83,5,0.573,-3.74,5,,Article 84 Penalties 1. Member States shall lay down the rules on other penalties applicable to infringements …
5,177,Recital 150,28,6,0.571,-5.20,6,,"infringements and the upper limit and criteria for setting the related administrative fines, which should be d…"
6,412,Article 83,83,9,0.534,-6.64,7,,"6. Non-compliance with an order by the supervisory authority as referred to in Article 58(2) shall, in accorda…"
7,176,Recital 149 – Recital 150,27,4,0.578,-8.35,8,,(149) Member States should be able to lay down the rules on criminal penalties for infringements of this Regu…
8,413,Article 83 – Article 84,83,7,0.566,-9.57,9,,8. The exercise by the supervisory authority of its powers under this Article shall be subject to appropriate …
9,179,Recital 151 – Recital 152,28,10,0.531,-9.64,10,,(151) The legal systems of Denmark and Estonia do not allow for administrative fines as set out in this Regul…


Critical chunk 411 (Article 83): Rank #8 → #1

Top-3 given to LLM  | baseline : ['410 (Article 83)', '175 (Recital 148)', '408 (Article 83)']
                    | re-ranked: ['411 (Article 83)', '410 (Article 83)', '408 (Article 83)']

--- Answer 1 (baseline) ---
The maximum fine for violating the basic principles of processing is 10,000,000 EUR or 2% of the annual global revenue, whichever is greater. This can be found in [1] under point (a).

--- Answer 2 (re-ranked) ---
The maximum fine for violating the basic principles of processing is 20,000,000 EUR or 4% of the annual global turnover, whichever is greater. This can be found in [1] under (a).


In [15]:
show(failure_results[5], failure_results[5]['gold'], failure_results[5]['category'])

[Counter-example: re-ranking hurts]  Q: When does the right to be forgotten not apply?


,chunk,section,page,bi_rank,bi_score,ce_score,ce_rank,★,text
0,71,Recital 65,12,2,0.532,2.02,1,,(65) A data subject should have the right to have personal data concerning him or her rectified and a ‘right t…
1,73,Recital 66,13,3,0.520,1.84,2,,"(66) To strengthen the right to be forgotten in the online environment, the right to erasure should also be ex…"
2,251,Article 15 – Article 17,43,1,0.549,1.23,3,,Section 3 Rectification and erasure Article 16 Right to rectification The data subject shall have the right to…
3,76,Recital 68,13,5,0.421,-4.11,4,,in the exercise of their public duties. It should therefore not apply where the processing of the personal dat…
4,258,Article 20 – Article 21,45,7,0.388,-5.84,5,,"2. In exercising his or her right to data portability pursuant to paragraph 1, the data subject shall have the…"
5,254,Article 17 – Article 18,44,4,0.435,-6.97,6,★,3. Paragraphs 1 and 2 shall not apply to the extent that processing is necessary: (a) for exercising the righ…
6,189,Recital 158,29,10,0.364,-7.34,7,,"(158) Where personal data are processed for archiving purposes, this Regulation should also apply to that pro…"
7,37,Recital 39,7,9,0.367,-8.98,8,,and rights in relation to the processing of personal data and how to exercise their rights in relation to such…
8,421,Article 89 – Article 90,85,6,0.393,-9.42,9,,"3. Where personal data are processed for archiving purposes in the public interest, Union or Member State law …"
9,256,Article 18 – Article 20,44,8,0.372,-10.27,10,,"2. Where processing has been restricted under paragraph 1, such personal data shall, with the exception of sto…"


Critical chunk 254 (Article 17 – Article 18): Rank #4 → #6

Top-3 given to LLM  | baseline : ['251 (Article 15 – Article 17)', '71 (Recital 65)', '73 (Recital 66)']
                    | re-ranked: ['71 (Recital 65)', '73 (Recital 66)', '251 (Article 15 – Article 17)']

--- Answer 1 (baseline) ---
The right to be forgotten does not apply when the personal data are no longer necessary in relation to the purposes for which they were collected or otherwise processed [1].

--- Answer 2 (re-ranked) ---
The right to be forgotten does not apply when the personal data are no longer necessary in relation to the purposes for which they were collected or otherwise processed, or where the processing of personal data does not otherwise comply with the GDPR [3].


In [16]:
show(failure_results[6], failure_results[6]['gold'], failure_results[6]['category'])

[Counter-example: first-stage recall ceiling]  Q: Can public authorities rely on legitimate interests as a legal basis?


,chunk,section,page,bi_rank,bi_score,ce_score,ce_rank,★,text
0,47,Recital 47,9,3,0.440,6.12,1,,that processing for that purpose may take place. The interests and fundamental rights of the data subject coul…
1,46,Recital 47,9,1,0.484,3.28,2,,"(47) The legitimate interests of a controller, including those of a controller to which the personal data may …"
2,27,Recital 31,6,9,0.354,0.24,3,,(31) Public authorities to which personal data are disclosed in accordance with a legal obligation for the exe…
3,189,Recital 158,29,5,0.383,-0.01,4,,"(158) Where personal data are processed for archiving purposes, this Regulation should also apply to that pro…"
4,78,Recital 69 – Recital 70,13,4,0.394,-0.54,5,,(69) Where personal data might lawfully be processed because processing is necessary for the performance of a …
5,61,Recital 55 – Recital 56,11,2,0.472,-0.54,6,,"(55) Moreover, the processing of personal data by official authorities for the purpose of achieving the aims, …"
6,45,Recital 46,8,7,0.376,-0.96,7,,(46) The processing of personal data should also be regarded to be lawful where it is necessary to protect an …
7,182,Recital 154,28,6,0.381,-1.67,8,,(154) This Regulation allows the principle of public access to official documents to be taken into account whe…
8,17,Recital 19,4,10,0.354,-2.60,9,,"private bodies falls within the scope of this Regulation, this Regulation should provide for the possibility f…"
9,44,Recital 45,8,8,0.367,-2.81,10,,"Regulation governing the lawfulness of personal data processing, establish specifications for determining the …"


Critical chunk 215 (Article 6): Rank #90 → not in top-10

Top-3 given to LLM  | baseline : ['46 (Recital 47)', '61 (Recital 55 – Recital 56)', '47 (Recital 47)']
                    | re-ranked: ['47 (Recital 47)', '46 (Recital 47)', '27 (Recital 31)']

--- Answer 1 (baseline) ---
Public authorities can rely on legitimate interests as a legal basis for processing personal data, but this must be assessed carefully considering the data subject's expectations and fundamental rights [1]. However, the processing by public authorities in the performance of their tasks does not fall under this legal basis as it is for the legislator to provide such a basis by law [3].

--- Answer 2 (re-ranked) ---
Based on Recital 47 [1], [2], public authorities can rely on legitimate interests as a legal basis for processing personal data, provided that the interests or fundamental rights of the data subject are not overriding. However, this must be assessed considering the reasonable expectations of data su

### 5.1 Rank-shift summary

In [17]:
rows = []
for res in failure_results:
    for s in gold_shift(res, res["gold"]):
        rows.append({"category": res["category"], "query": res["query"], "critical chunk": s["chunk"],
                     "section": s["section"], "baseline rank": s["baseline_rank"],
                     "re-ranked rank": s["reranked_rank"] if s["reranked_rank"] else "not in top-10",
                     "in LLM context (P1)": s["baseline_rank"] <= TOP_K_CONTEXT,
                     "in LLM context (P2)": bool(s["reranked_rank"] and s["reranked_rank"] <= TOP_K_CONTEXT)})
shift_df = pd.DataFrame(rows)
shift_df.to_csv(RESULTS_DIR / "rank_shifts.csv", index=False)
shift_df

,category,query,critical chunk,section,baseline rank,re-ranked rank,in LLM context (P1),in LLM context (P2)
0,Keyword trap / distractor,How much can a company be fined for failing to report a data breach?,410,Article 83,7,3,False,True
1,Exception / corner case,When is a controller not required to tell affected individuals about a personal data breach?,288,Article 34,5,2,False,True
2,Negation / prohibition,Is it forbidden to process biometric data to identify a person?,223,Article 8 – Article 9,8,2,False,True
3,Role limit (synonyms + negation),Can a member of a supervisory authority have another paid job?,142,Recital 121,1,2,True,True
4,Role limit (synonyms + negation),Can a member of a supervisory authority have another paid job?,342,Article 51 – Article 52,10,7,False,False
5,Role limit (synonyms + negation),Can a member of a supervisory authority have another paid job?,343,Article 52 – Article 53,7,1,False,True
6,Buried chunk (rank 4-10),What is the maximum fine for violating the basic principles of processing?,411,Article 83,8,1,False,True
7,Counter-example: re-ranking hurts,When does the right to be forgotten not apply?,254,Article 17 – Article 18,4,6,False,False
8,Counter-example: first-stage recall ceiling,Can public authorities rely on legitimate interests as a legal basis?,215,Article 6,90,not in top-10,False,False


In [18]:
# Save both answers for every failure query (used when writing the report)
with open(RESULTS_DIR / "answers.json", "w") as f:
    json.dump([{"category": r["category"], "query": r["query"],
                "top3_baseline": [f"{int(i)} ({chunks[i]['section']})" for i in r["base"]["idx"][:3]],
                "top3_reranked": [f"{int(i)} ({chunks[i]['section']})" for i in r["rr"]["idx"][:3]],
                "answer1": r["answer1"], "answer2": r["answer2"]} for r in failure_results], f, indent=2, ensure_ascii=False)
print("saved results/answers.json")

saved results/answers.json


## 6. Retrieval quality on a labelled query set

To check that the stress tests are not cherry-picked in isolation, here is a small labelled set (the 5 stress-test queries plus
15 ordinary questions, each with a regex for its critical chunk). Metrics: **Hit@3** (is a critical chunk in the 3 chunks that
reach the LLM?) and **MRR@10**. Because both pipelines re-order the *same* 10 candidates, Recall@10 is identical by construction.

In [19]:
EVAL_SET = [(q, p) for _, q, p in FAILURE_QUERIES[:5]] + [
    ("How long does a company have to report a data breach to the regulator?", r"not later than 72 hours"),
    ("Can a data protection officer be fired for doing their job?", r"shall not be dismissed or penalised"),
    ("At what age can a child consent on their own to an online service?", r"at least 16 years"),
    ("Can I charge a fee when someone asks for a copy of their data?", r"reasonable fee"),
    ("Does the regulation apply to a person keeping an address book for personal use?", r"purely personal or household activity"),
    ("Can the data protection officer be an external contractor?", r"service contract"),
    ("Can a person be subject to a decision made entirely by an algorithm?", r"decision based solely on automated processing"),
    ("Can consent be withdrawn and does that make past processing unlawful?", r"withdrawal of consent shall not affect the lawfulness"),
    ("How quickly must a company answer a request from an individual exercising their rights?", r"within one month of receipt of the request"),
    ("Can the data protection officer be told by management how to carry out their tasks?", r"does not receive any instructions"),
    ("When can a controller refuse to act on a data subject's request?", r"manifestly unfounded or excessive"),
    ("Is a processor liable for damage caused by processing?", r"processor shall be liable for the damage"),
    ("Can a processor hire a subcontractor without permission from the controller?", r"shall not engage another processor without prior"),
    ("Is a certification proof of compliance that frees the controller from responsibility?", r"does not reduce the responsibility"),
    ("How long is a supervisory authority member's term of office?", r"no less than four years"),
]

def first_hit(ranked, gold):
    for r, i in enumerate(ranked):
        if int(i) in gold:
            return r + 1
    return None

eval_rows = []
for q, pat in EVAL_SET:
    gold = set(gold_ids(pat))
    res = run_both(q, with_llm=False)
    b, c = first_hit(res["base"]["idx"], gold), first_hit(res["rr"]["idx"], gold)
    eval_rows.append({"query": q, "baseline first-hit rank": b, "re-ranked first-hit rank": c})
eval_df = pd.DataFrame(eval_rows)

def hit3(col): return (eval_df[col].fillna(99) <= 3).mean()
def mrr(col):  return eval_df[col].apply(lambda r: 0 if pd.isna(r) else 1 / r).mean()
summary = pd.DataFrame({
    "Pipeline": ["1: Bi-encoder top-3", "2: Cross-encoder re-ranked top-3"],
    "Hit@3": [hit3("baseline first-hit rank"), hit3("re-ranked first-hit rank")],
    "MRR@10": [mrr("baseline first-hit rank"), mrr("re-ranked first-hit rank")],
    "Recall@10": [eval_df["baseline first-hit rank"].notna().mean()] * 2,
}).round(3)
eval_df.to_csv(RESULTS_DIR / "eval_ranks.csv", index=False)
summary.to_csv(RESULTS_DIR / "eval_summary.csv", index=False)
display(eval_df); summary

,query,baseline first-hit rank,re-ranked first-hit rank
0,How much can a company be fined for failing to report a data breach?,7,3
1,When is a controller not required to tell affected individuals about a personal data breach?,5,2
2,Is it forbidden to process biometric data to identify a person?,8,2
3,Can a member of a supervisory authority have another paid job?,1,1
4,What is the maximum fine for violating the basic principles of processing?,8,1
5,How long does a company have to report a data breach to the regulator?,3,1
6,Can a data protection officer be fired for doing their job?,1,1
7,At what age can a child consent on their own to an online service?,1,1
8,Can I charge a fee when someone asks for a copy of their data?,1,1
9,Does the regulation apply to a person keeping an address book for personal use?,1,1


,Pipeline,Hit@3,MRR@10,Recall@10
0,1: Bi-encoder top-3,0.8,0.713,1.0
1,2: Cross-encoder re-ranked top-3,1.0,0.867,1.0


## 7. Latency profiling

Every stage is timed with `time.perf_counter()` (with `torch.cuda.synchronize()` around GPU work), after a warm-up,
over **all 20 labelled queries × 10 repetitions = 200 runs**. LLM generation time is reported separately because it is
identical in structure for both pipelines (same prompt template, 3 chunks). Any difference in LLM time between the two
pipelines comes from how many tokens the model chose to generate, so tokens generated and ms/token are reported too.

In [20]:
REPS = 10
lat = {"embed": [], "search": [], "baseline_total": [], "rerank": []}
for _ in range(REPS):
    for q, _ in EVAL_SET:
        b = baseline_retrieve(q)
        r = rerank(q, b["idx"])
        lat["embed"].append(b["t_embed"]); lat["search"].append(b["t_search"])
        lat["baseline_total"].append(b["t_total"]); lat["rerank"].append(r["t_rerank"])
ms = {k: np.array(v) * 1000 for k, v in lat.items()}
p2_total = ms["baseline_total"] + ms["rerank"]

def stats(a): return [a.mean(), np.median(a), np.percentile(a, 95), a.std()]
lat_df = pd.DataFrame(
    [["Pipeline 1", "Query embedding (bi-encoder)"] + stats(ms["embed"]),
     ["Pipeline 1", "Vector search (430 chunks, top-10)"] + stats(ms["search"]),
     ["Pipeline 1", "TOTAL baseline retrieval"] + stats(ms["baseline_total"]),
     ["Pipeline 2", "Cross-encoder re-rank (10 pairs)"] + stats(ms["rerank"]),
     ["Pipeline 2", "TOTAL retrieval + re-rank"] + stats(p2_total)],
    columns=["pipeline", "stage", "mean ms", "median ms", "p95 ms", "std ms"]).round(2)

llm_t = np.array([[r["t_answer1"], r["t_answer2"]] for r in failure_results]) * 1000
llm_n = np.array([[r["n_answer1"], r["n_answer2"]] for r in failure_results])
llm_df = pd.DataFrame({"pipeline": ["Pipeline 1", "Pipeline 2"],
                       "LLM generation mean ms": llm_t.mean(0),
                       "mean tokens generated": llm_n.mean(0),
                       "ms per generated token": (llm_t / llm_n).mean(0),
                       "end-to-end mean ms (retrieval + LLM)": [ms["baseline_total"].mean() + llm_t[:, 0].mean(),
                                                                 p2_total.mean() + llm_t[:, 1].mean()]}).round(1)
print(f"Re-ranking overhead: +{ms['rerank'].mean():.2f} ms per query "
      f"({p2_total.mean() / ms['baseline_total'].mean():.1f}x the baseline retrieval time), device={DEVICE}")
lat_df.to_csv(RESULTS_DIR / "latency.csv", index=False); llm_df.to_csv(RESULTS_DIR / "latency_llm.csv", index=False)
display(lat_df); llm_df

Re-ranking overhead: +13.48 ms per query (5.2x the baseline retrieval time), device=cuda


,pipeline,stage,mean ms,median ms,p95 ms,std ms
0,Pipeline 1,Query embedding (bi-encoder),3.11,2.89,4.48,0.62
1,Pipeline 1,"Vector search (430 chunks, top-10)",0.06,0.06,0.08,0.01
2,Pipeline 1,TOTAL baseline retrieval,3.17,2.94,4.55,0.63
3,Pipeline 2,Cross-encoder re-rank (10 pairs),13.48,13.39,15.15,0.88
4,Pipeline 2,TOTAL retrieval + re-rank,16.66,16.38,19.57,1.20


,pipeline,LLM generation mean ms,mean tokens generated,ms per generated token,end-to-end mean ms (retrieval + LLM)
0,Pipeline 1,1519.7,47.0,32.6,1522.9
1,Pipeline 2,2112.6,66.9,33.1,2129.2


## 8. Scaling experiment: why not cross-encode the whole corpus?

To support the scaling question with numbers, I time the cross-encoder on (a) the 10 candidates and (b) **all 430 chunks**
of this single document, and compare with the bi-encoder search over all 430 chunks. Cross-encoder cost grows linearly with the
number of documents *per query* because nothing can be pre-computed; the bi-encoder's document side is pre-computed once.

In [21]:
def time_ce(n, q="What is the maximum fine for violating the basic principles of processing?", reps=5):
    pairs = [(q, chunk_texts[i % len(chunk_texts)]) for i in range(n)]
    cross_encoder.predict(pairs[:10], batch_size=64)              # warm-up
    ts = []
    for _ in range(reps):
        sync(); t0 = time.perf_counter()
        cross_encoder.predict(pairs, batch_size=64)
        sync(); ts.append(time.perf_counter() - t0)
    return np.mean(ts) * 1000

scale_rows = [{"setting": "Bi-encoder: embed query + search all 430 chunks", "ms per query": ms["baseline_total"].mean()},
              {"setting": "Cross-encoder: 10 candidates", "ms per query": time_ce(10)},
              {"setting": f"Cross-encoder: all {len(chunks)} chunks (whole document)", "ms per query": time_ce(len(chunks))}]
scale_df = pd.DataFrame(scale_rows).round(1)
per_pair = scale_df.iloc[2]["ms per query"] / len(chunks)
print(f"≈ {per_pair:.3f} ms per (query, chunk) pair on {DEVICE}; extrapolated to 1M chunks ≈ {per_pair * 1e6 / 1000 / 60:.0f} min per query")

# Does exhaustive cross-encoding even rank better? Check the stress-test queries.
rows = []
for cat, q, pat in FAILURE_QUERIES:
    gold = set(gold_ids(pat))
    s = cross_encoder.predict([(q, t) for t in chunk_texts], batch_size=64)
    rows.append({"category": cat, "rank of critical chunk with CE over all chunks": first_hit(np.argsort(-s), gold)})
scale_df.to_csv(RESULTS_DIR / "scaling.csv", index=False)
display(scale_df); pd.DataFrame(rows)

≈ 1.106 ms per (query, chunk) pair on cuda; extrapolated to 1M chunks ≈ 18 min per query


,setting,ms per query
0,Bi-encoder: embed query + search all 430 chunks,3.2
1,Cross-encoder: 10 candidates,13.1
2,Cross-encoder: all 430 chunks (whole document),475.6


,category,rank of critical chunk with CE over all chunks
0,Keyword trap / distractor,3
1,Exception / corner case,2
2,Negation / prohibition,2
3,Role limit (synonyms + negation),1
4,Buried chunk (rank 4-10),1
5,Counter-example: re-ranking hurts,20
6,Counter-example: first-stage recall ceiling,4


## 9. Where to find the outputs
* `results/rank_shifts.csv` – rank change of every critical chunk for the stress-test queries
* `results/answers.json` – Answer 1 vs Answer 2 for each stress-test query, with the top-3 chunks each pipeline used
* `results/eval_summary.csv`, `results/eval_ranks.csv` – Hit@3 / MRR on the labelled set
* `results/latency.csv`, `results/latency_llm.csv`, `results/scaling.csv` – timing tables